# Clonar voz para el comercial de AutomLead

Genera los 8 segmentos de locución con **Chatterbox** (Resemble AI, licencia MIT),
clonando la voz que vos subas.

Corre en la **GPU gratuita de Colab**. Tarda unos 10 minutos en total.

---

## Antes de empezar: activá la GPU

En el menú de arriba: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.

Sin eso no funciona (o tarda horas).

## Qué necesitás tener a mano

Un archivo de audio con la voz a clonar:

- **15 a 30 segundos** es el punto justo. Menos no alcanza; más no mejora.
- **Limpio**: sin música de fondo, sin ruido, sin otra persona hablando.
- **Hablando normal**, en el tono en que querés que suene el comercial.
- `.wav` o `.mp3`, grabado con el celular alcanza.

> Cuanto mejor la muestra, mejor el resultado. Una muestra con eco o ruido
> se clona **con** el eco y el ruido.

## 1. Verificar que la GPU está activa

Si esta celda dice que no hay GPU, volvé arriba y activala antes de seguir.

In [ ]:
import subprocess, sys
salida = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if salida.returncode != 0:
    print('SIN GPU. Entorno de ejecucion -> Cambiar tipo -> T4 GPU -> Guardar')
    print('Despues volve a correr esta celda.')
else:
    for linea in salida.stdout.splitlines():
        if 'MiB' in linea or 'Tesla' in linea or 'T4' in linea:
            print(linea.strip())
    print('\nGPU lista.')

## 2. Instalar Chatterbox

Tarda 2-3 minutos. Es normal que imprima muchas líneas y algún aviso de
versiones: mientras termine, está bien.

In [ ]:
!pip install -q chatterbox-tts
print('Instalado.')

## 3. Subir la muestra de voz

Al correr esta celda aparece un botón **Elegir archivos**. Subí tu grabación.

In [ ]:
from google.colab import files
import os

subidos = files.upload()
REFERENCIA = list(subidos.keys())[0]
print('\nMuestra:', REFERENCIA, '(%.0f KB)' % (os.path.getsize(REFERENCIA)/1024))

# Chatterbox espera wav mono a 24 kHz: lo convertimos por las dudas
!ffmpeg -v error -i "$REFERENCIA" -ar 24000 -ac 1 -y referencia.wav
REFERENCIA = 'referencia.wav'

import wave
with wave.open(REFERENCIA) as w:
    dur = w.getnframes() / w.getframerate()
print('Duracion: %.1f s' % dur)
if dur < 8:
    print('AVISO: es corta. Con menos de 8 s el clonado sale flojo.')
elif dur > 45:
    print('AVISO: es larga. Con 15-30 s alcanza y anda mejor.')
else:
    print('Duracion adecuada.')

## 4. Escuchar la referencia

Confirmá que se escucha limpio y que es el fragmento que querés clonar.

In [ ]:
from IPython.display import Audio, display
display(Audio(REFERENCIA))

## 5. Cargar el modelo

La primera vez descarga unos 2 GB de pesos. Tarda 2-4 minutos.

In [ ]:
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

modelo = ChatterboxMultilingualTTS.from_pretrained(device='cuda', t3_model='v3')
print('Modelo cargado. Sample rate:', modelo.sr)

## 6. El guion

Son los 8 segmentos del comercial. **No cambies los `id`**: el montaje los
busca por ese nombre exacto.

El texto sí lo podés editar si querés ajustar algo.

In [ ]:
GUION = [
    ('01-gancho',
     'Sábado, once y cuarenta y siete de la noche. Alguien pregunta por una camioneta que tenés publicada.'),
    ('02-pregunta',
     '¿Cuánto tarda tu concesionaria en contestar ese mensaje?'),
    ('03-espera',
     'Hoy queda en el celular de alguien. Y nadie lo ve hasta el lunes a las ocho de la mañana.'),
    ('04-golpe',
     'Treinta y dos horas y trece minutos. Para entonces ese cliente ya está hablando con otra concesionaria.'),
    ('05-automlead',
     'Con AutomLead, tres segundos. Respondido, cualificado, y con un vendedor encima.'),
    ('06-regla',
     'Todo el sistema se apoya en una sola regla: un lead no puede quedar sin dueño. Si ningún vendedor lo toma, se asigna igual.'),
    ('07-capacidades',
     'Contesta a los tres segundos, siempre. Puntúa cada consulta con una cuenta que podés auditar. Y te muestra cuánto tarda de verdad cada vendedor en responder.'),
    ('08-cierre',
     '¿Cuántas consultas se te enfriaron este mes sin que te enteres? Agendá tu diagnóstico. Funciona sobre tu número de WhatsApp actual.'),
]

print('%d segmentos, %d caracteres en total'
      % (len(GUION), sum(len(t) for _, t in GUION)))

## 7. Generar

Unos 5 minutos. Va imprimiendo cada segmento con su duración.

**Los dos controles que importan:**

- `exaggeration` — cuánta expresividad. `0.5` es neutro de locutor.
  Subilo a `0.7` si lo querés más vendedor; bajalo a `0.3` para más sobrio.
- `cfg_weight` — cuánto se pega a la referencia. `0.5` equilibra.
  Más alto imita más pero puede sonar rígido.

In [ ]:
import os, time

EXAGGERATION = 0.5
CFG_WEIGHT = 0.5

os.makedirs('voz', exist_ok=True)
duraciones = []
t_total = time.time()

for sid, texto in GUION:
    t0 = time.time()
    wav = modelo.generate(
        texto,
        language_id='es',
        audio_prompt_path=REFERENCIA,
        exaggeration=EXAGGERATION,
        cfg_weight=CFG_WEIGHT,
    )
    destino = f'voz/{sid}.wav'
    ta.save(destino, wav, modelo.sr)
    dur = wav.shape[-1] / modelo.sr
    duraciones.append((sid, dur))
    print('%-16s %5.2f s de audio  (%.0fs de computo)' % (sid, dur, time.time()-t0))

total = sum(d for _, d in duraciones)
print('\nTOTAL LOCUCION: %.2f s   |   computo: %.0fs' % (total, time.time()-t_total))

## 8. Escuchar antes de bajar

Si no te convence, volvé a la celda 7, cambiá `EXAGGERATION` y corré de nuevo.
No hace falta recargar el modelo.

In [ ]:
from IPython.display import Audio, display
import IPython.display as ipd

for sid, _ in duraciones[:3]:
    print(sid)
    display(Audio(f'voz/{sid}.wav'))

## 9. Descargar

Baja un `voz-clonada.zip`. **Mandámelo tal cual** y yo lo integro:
recalculo los tiempos de cada escena según las duraciones nuevas,
regenero la música para que los acentos sigan cayendo en los cortes,
y renderizo.

In [ ]:
import json

# Las duraciones viajan con el audio: las necesito para rearmar el montaje
with open('voz/duraciones.json', 'w', encoding='utf-8') as f:
    json.dump([{'id': s, 'segundos': round(d, 3)} for s, d in duraciones],
              f, indent=2, ensure_ascii=False)

!cd voz && zip -q -r ../voz-clonada.zip .

from google.colab import files
files.download('voz-clonada.zip')
print('Listo. Mandame el zip.')

---

## Si algo sale mal

| Síntoma | Qué hacer |
| --- | --- |
| `CUDA out of memory` | Entorno de ejecución → Reiniciar, y corré todo de nuevo |
| Suena robótico o metálico | La muestra tiene ruido o eco. Grabá de nuevo en un lugar silencioso |
| No respeta el acento | Subí `CFG_WEIGHT` a `0.7` |
| Suena plano, sin intención | Subí `EXAGGERATION` a `0.7` |
| Se come palabras | Cortá el segmento en dos frases más cortas |
| Colab se desconecta | El plan gratuito corta por inactividad. Volvé a correr desde la celda 5 |

**Licencia:** Chatterbox es MIT. Lo que generes es tuyo y lo podés usar
comercialmente. El modelo inserta una marca de agua inaudible en el audio,
que es trazabilidad de origen y no restringe el uso.